# Run MyCF

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the project `my_cf` / `cf_metric_opt_upper` counterfactual explainer through the shared benchmark pipeline.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_metric_optimized_cf_upper/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("01_my_cf.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root


In [2]:
import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODEL_TYPES = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODEL_TYPES)}")

CTX = prepare_explainer_run("01_my_cf.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_TYPE = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

N_EVAL_EVENTS = int(SETTINGS["n_eval_events"])
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
TRIAL_CONFIGS = [dict(item) for item in SETTINGS["trial_configs"]]

CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device
out_dir = BENCHMARK_ROOT / "resources" / "results" / str(CFG.get("out_dir_name", "official_metric_optimized_cf_upper"))
out_dir.mkdir(parents=True, exist_ok=True)

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_TYPE)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: ucim
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/ucim/tgn/tgn_ucim_best.pth
DEVICE : mps


In [3]:
# Load CoDy baseline run (summary + JSONL) and anchor events.
import json
from typing import Any

import numpy as np
import pandas as pd

from I_explainer_benchmark.src.explainers.extractors.khop import KHopCandidatesExtractor
from I_explainer_benchmark.src.models.loader import load_backbone_model
from I_explainer_benchmark.src.core.runner import EvalConfig, EvaluationRunner
from I_explainer_benchmark.src.core.types import (
    BaseExplainer,
    ExplanationContext,
    ExplanationResult,
)
from I_explainer_benchmark.src.viz.visualization.metrics import compute_cody_style_report
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo

cody_root = REPO_ROOT / "resources" / "results" / "official_cody"
if not cody_root.exists():
    raise FileNotFoundError(f"Missing CoDy results directory: {cody_root}")

combo_prefix = f"{DATASET_NAME}_{MODEL_TYPE}_official_cody_"
summary_candidates = sorted(
    cody_root.glob(f"{combo_prefix}*_cody_metrics_summary.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not summary_candidates:
    raise FileNotFoundError(
        f"No CoDy summary CSV found for dataset={DATASET_NAME!r}, model={MODEL_TYPE!r} in {cody_root}."
    )

baseline_pairs = []
for summary_path in summary_candidates:
    base_name = summary_path.name.replace("_cody_metrics_summary.csv", "")
    jsonl_path = cody_root / f"{base_name}.jsonl"
    if jsonl_path.exists():
        baseline_pairs.append((summary_path, jsonl_path, base_name))


baseline_summary_path, baseline_jsonl_path, baseline_base_name = baseline_pairs[0]
baseline_df = pd.read_csv(baseline_summary_path)
if baseline_df.empty or "explainer" not in baseline_df.columns:
    raise RuntimeError(f"Baseline CoDy summary is empty or missing explainer column: {baseline_summary_path}")

rows = baseline_df[baseline_df["explainer"].astype(str).str.lower() == "cody"]
if rows.empty:
    raise RuntimeError(f"Baseline CoDy summary does not contain explainer='cody': {baseline_summary_path}")

baseline_row = rows.iloc[-1].to_dict()

with baseline_jsonl_path.open("r", encoding="utf-8") as f:
    baseline_records = [json.loads(line) for line in f if line.strip()]

baseline_map: dict[int, dict[str, Any]] = {}
ordered_event_idxs: list[int] = []
for rec in baseline_records:
    result = rec.get("result") or {}
    if str(result.get("explainer", "")).lower() != "cody":
        continue
    context = rec.get("context") or {}
    target = context.get("target") or {}
    extras = result.get("extras") or {}

    event_idx = target.get("event_idx")
    if event_idx is None:
        event_idx = extras.get("event_idx")
    if event_idx is None:
        continue
    event_idx = int(event_idx)

    selected = extras.get("cf_event_ids")
    if selected is None:
        selected = extras.get("selected_eidx")
    selected_ids = [int(x) for x in list(selected or [])]

    baseline_map[event_idx] = {
        "selected": selected_ids,
        "original_prediction": extras.get("original_prediction"),
        "counterfactual_prediction": extras.get("counterfactual_prediction"),
    }
    ordered_event_idxs.append(event_idx)

if not ordered_event_idxs:
    raise RuntimeError(f"Baseline CoDy JSONL did not contain event-indexed records: {baseline_jsonl_path}")

ordered_unique: list[int] = []
seen: set[int] = set()
for eidx in ordered_event_idxs:
    if eidx in seen:
        continue
    seen.add(eidx)
    ordered_unique.append(int(eidx))

FORCED_TARGET_EVENT_IDS = list(globals().get("FORCED_TARGET_EVENT_IDS", _forced_target_event_ids_for_combo(DATASET_NAME, MODEL_TYPE)))
_target_budget = int(min(max(1, N_EVAL_EVENTS), len(ordered_unique)))
target_event_idxs: list[int] = []
_seen_target_event_idxs: set[int] = set()
for _event_id in [*FORCED_TARGET_EVENT_IDS, *ordered_unique]:
    _event_id = int(_event_id)
    if _event_id in _seen_target_event_idxs:
        continue
    target_event_idxs.append(_event_id)
    _seen_target_event_idxs.add(_event_id)
    if len(target_event_idxs) >= _target_budget:
        break
anchors = [{"target_kind": "edge", "event_idx": int(e)} for e in target_event_idxs]

baseline_plus = float(baseline_row.get("cody_AUFSC_plus", np.nan))
baseline_charr = float(baseline_row.get("cody_CHARR", np.nan))

baseline_best_fid = np.nan
baseline_aufsc = np.nan
baseline_best_fid_raw = np.nan
baseline_best_fid_sparsity = np.nan
baseline_tgnn_summary_path = (
    cody_root
    / f"{baseline_base_name}_tgnn_metrics"
    / f"{baseline_base_name}_tgnn_aufsc_bestfid_summary_both.csv"
)
if baseline_tgnn_summary_path.exists():
    tgnn_base = pd.read_csv(baseline_tgnn_summary_path)
    row = tgnn_base[
        (tgnn_base.get("explainer", pd.Series(dtype=str)).astype(str) == "cody")
        & (tgnn_base.get("variant", pd.Series(dtype=str)).astype(str) == "official")
    ]
    if not row.empty:
        r = row.iloc[-1]
        baseline_best_fid = float(r.get("best_fid", np.nan))
        baseline_aufsc = float(r.get("aufsc", np.nan))
        baseline_best_fid_raw = float(r.get("best_fid_raw", np.nan))
        baseline_best_fid_sparsity = float(r.get("best_fid_sparsity", np.nan))

print("Baseline summary:", baseline_summary_path)
print("Baseline jsonl  :", baseline_jsonl_path)
print("Baseline TGNN summary:", baseline_tgnn_summary_path if baseline_tgnn_summary_path.exists() else "missing")
print("Baseline CoDy drop targets:")
print("  cody_AUFSC_plus:", baseline_plus)
print("  cody_CHARR     :", baseline_charr)
print("Baseline TGNN targets (official):")
print("  best_fid          :", baseline_best_fid)
print("  aufsc             :", baseline_aufsc)
print("  best_fid_raw      :", baseline_best_fid_raw)
print("  best_fid_sparsity :", baseline_best_fid_sparsity)
print("Anchors selected :", len(anchors))


Baseline summary: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_cody/ucim_tgn_official_cody_20260315_164433_cody_metrics_summary.csv
Baseline jsonl  : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_cody/ucim_tgn_official_cody_20260315_164433.jsonl
Baseline TGNN summary: missing
Baseline CoDy drop targets:
  cody_AUFSC_plus: 1.3555271928274224
  cody_CHARR     : 0.7867860001437902
Baseline TGNN targets (official):
  best_fid          : nan
  aufsc             : nan
  best_fid_raw      : nan
  best_fid_sparsity : nan
Anchors selected : 30


In [4]:
# Load model + data and define the metric-optimized counterfactual explainer.
import time
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

model, events = load_backbone_model(
    model_type=MODEL_TYPE,
    dataset_name=DATASET_NAME,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
)
backbone = model.backbone

extractor_for_eval = KHopCandidatesExtractor(
    model=model,
    events=events,
    candidates_size=CANDIDATES_SIZE,
    num_hops=int(getattr(model, "num_layers", 2) or 2),
)


def _as_scalar_score(value: Any) -> float:
    if isinstance(value, torch.Tensor):
        if value.ndim == 0:
            return float(value.item())
        return float(value.reshape(-1)[0].item())
    if isinstance(value, (np.ndarray, list, tuple)):
        arr = np.asarray(value).reshape(-1)
        return float(arr[0]) if arr.size else float("nan")
    return float(value)


def _cody_delta_ratio(original_score: float, perturbed_score: float) -> float:
    orig = float(original_score)
    pert = float(perturbed_score)
    denom = max(abs(orig), 1e-12)
    if orig * pert < 0:
        delta = abs(orig) + abs(pert)
    else:
        delta = abs(orig) - abs(pert)
    return max(0.0, float(delta) / float(denom))


def _progress_to_boundary(original_score: float, perturbed_score: float) -> float:
    """Normalized progress toward/over decision boundary (higher is better)."""
    orig = float(original_score)
    pert = float(perturbed_score)
    denom = max(abs(orig), 1e-12)

    # Signed margin on original label side: positive means still same side, negative means flipped.
    if orig >= 0.0:
        signed = pert
    else:
        signed = -pert

    if signed < 0.0:
        # Already crossed boundary: reward both crossing and confidence beyond boundary.
        return float((abs(orig) + abs(pert)) / denom)
    # Not crossed yet: reward shrinking the signed margin.
    return float(max(0.0, abs(orig) - signed) / denom)


@dataclass
class MetricOptimizedCFConfig:
    alias: str = "cf_metric_opt_upper"
    mode: str = "beam"  # beam | cody_clone
    beam_width: int = 8
    max_steps: int = 10
    pool_size: int = 30
    use_cody_floor: bool = True
    stop_at_first_flip: bool = False
    sparsity_penalty: float = 0.05
    cody_seed_bonus: float = 0.30
    progress_weight: float = 0.35
    cody_map: dict[int, dict[str, Any]] | None = None


class MetricOptimizedCounterfactualExplainer(BaseExplainer):
    def __init__(self, cfg: MetricOptimizedCFConfig) -> None:
        self.cfg = cfg
        super().__init__(name="metric_optimized_counterfactual", alias=str(cfg.alias))

    def prepare(self, *, model: Any, dataset: Any) -> None:
        super().prepare(model=model, dataset=dataset)

    def _event_idx(self, context: ExplanationContext) -> int | None:
        target = context.target if isinstance(context.target, dict) else {}
        v = target.get("event_idx") or target.get("index") or target.get("idx")
        if v is None and context.subgraph is not None and isinstance(context.subgraph.payload, dict):
            v = context.subgraph.payload.get("event_idx")
        try:
            return int(v) if v is not None else None
        except Exception:
            return None

    def _candidate_eidx(self, context: ExplanationContext) -> list[int]:
        payload = context.subgraph.payload if context.subgraph and isinstance(context.subgraph.payload, dict) else {}
        raw = payload.get("candidate_eidx")
        if raw is None:
            return []
        out = []
        for x in list(raw):
            try:
                out.append(int(x))
            except Exception:
                continue
        return out

    def _predict_drop(self, context: ExplanationContext, candidate: list[int], dropped: set[int]) -> float:
        edge_mask = [0.0 if int(eid) in dropped else 1.0 for eid in candidate]
        pred = self._model.predict_proba_with_mask(context.subgraph, context.target, edge_mask=edge_mask)
        return _as_scalar_score(pred)

    def _predict_keep_only(self, context: ExplanationContext, candidate: list[int], selected: set[int]) -> float:
        edge_mask = [1.0 if int(eid) in selected else 0.0 for eid in candidate]
        pred = self._model.predict_proba_with_mask(context.subgraph, context.target, edge_mask=edge_mask)
        return _as_scalar_score(pred)

    def explain(self, context: ExplanationContext) -> ExplanationResult:
        t0 = time.perf_counter()

        candidate = self._candidate_eidx(context)
        event_idx = self._event_idx(context)
        cody_row = (self.cfg.cody_map or {}).get(int(event_idx)) if event_idx is not None else None

        if not candidate:
            return ExplanationResult(
                run_id=context.run_id,
                explainer=self.alias,
                context_fp=context.fingerprint(),
                importance_edges=[],
                elapsed_sec=float(time.perf_counter() - t0),
                extras={
                    "event_idx": int(event_idx) if event_idx is not None else None,
                    "candidate_eidx": [],
                    "cf_event_ids": [],
                    "achieves_counterfactual_explanation": False,
                },
            )

        z_orig = None
        if isinstance(cody_row, dict):
            try:
                z_orig = float(cody_row.get("original_prediction"))
            except Exception:
                z_orig = None
        if z_orig is None or not np.isfinite(z_orig):
            z_orig = _as_scalar_score(self._model.predict_proba(context.subgraph, context.target))

        orig_label = int(float(z_orig) >= 0.0)
        n_candidate = max(1, int(len(candidate)))
        cody_selected = set(int(e) for e in list((cody_row or {}).get("selected") or []))

        def _is_flip(z_val: float) -> bool:
            return bool((float(z_val) >= 0.0) != bool(orig_label))

        def _state_key(drop_tuple: tuple[int, ...], z_drop: float, ratio: float, progress: float) -> tuple[float, float, float, float, float]:
            # Lexicographic objective:
            # 1) flip first, 2) minimize sparsity when flipped,
            # 3) maximize blended improvement, 4/5) tie-breakers.
            flip = float(int(_is_flip(z_drop)))
            sparsity = float(len(drop_tuple)) / float(n_candidate)
            blended = float(ratio) + float(self.cfg.progress_weight) * float(progress) - float(self.cfg.sparsity_penalty) * sparsity
            return (flip, -sparsity if flip > 0.5 else blended, blended, float(ratio), float(progress))

        selected_best: list[int] = []
        z_cf_best = float(z_orig)
        ratio_best = 0.0
        progress_best = 0.0
        best_key = _state_key(tuple(), float(z_orig), 0.0, 0.0)
        choice = "beam"

        if str(self.cfg.mode).lower() == "cody_clone" and isinstance(cody_row, dict):
            selected_cody = [int(e) for e in list(cody_row.get("selected") or []) if int(e) in set(candidate)]
            z_cf_cody = cody_row.get("counterfactual_prediction")
            if z_cf_cody is None or not np.isfinite(float(z_cf_cody)):
                z_cf_cody = self._predict_drop(context, candidate, set(selected_cody))
            z_cf_cody = float(z_cf_cody)

            ratio_cody = _cody_delta_ratio(z_orig, z_cf_cody)
            progress_cody = _progress_to_boundary(z_orig, z_cf_cody)
            selected_best = list(selected_cody)
            z_cf_best = z_cf_cody
            ratio_best = float(ratio_cody)
            progress_best = float(progress_cody)
            best_key = _state_key(tuple(sorted(selected_best)), z_cf_best, ratio_best, progress_best)
            choice = "cody_clone"
        else:
            # Rank candidates by single-edge drop effect + CoDy seed prior.
            single = []
            for eid in candidate:
                z_drop = self._predict_drop(context, candidate, {int(eid)})
                ratio = _cody_delta_ratio(z_orig, z_drop)
                progress = _progress_to_boundary(z_orig, z_drop)
                seed_bonus = float(self.cfg.cody_seed_bonus) if int(eid) in cody_selected else 0.0
                rank_score = float(ratio + float(self.cfg.progress_weight) * progress + seed_bonus)
                single.append((int(eid), float(z_drop), float(ratio), float(progress), rank_score))
            single.sort(key=lambda x: x[4], reverse=True)

            pool_ids = [eid for eid, *_ in single[: max(1, int(self.cfg.pool_size))]]
            if not pool_ids:
                pool_ids = list(candidate)

            cache: dict[tuple[int, ...], tuple[float, float, float]] = {}

            def _eval_drop(drop_tuple: tuple[int, ...]) -> tuple[float, float, float]:
                if drop_tuple in cache:
                    return cache[drop_tuple]
                z_drop = self._predict_drop(context, candidate, set(drop_tuple))
                ratio = _cody_delta_ratio(z_orig, z_drop)
                progress = _progress_to_boundary(z_orig, z_drop)
                cache[drop_tuple] = (float(z_drop), float(ratio), float(progress))
                return cache[drop_tuple]

            beam: list[tuple[int, ...]] = [tuple()]
            best = (tuple(), float(z_orig), 0.0, 0.0)

            for _ in range(max(1, int(self.cfg.max_steps))):
                expanded: list[tuple[tuple[int, ...], float, float, float, tuple[float, float, float, float, float]]] = []
                seen_states: set[tuple[int, ...]] = set()

                for state in beam:
                    state_set = set(state)
                    for eid in pool_ids:
                        if int(eid) in state_set:
                            continue
                        new_state = tuple(sorted([*state, int(eid)]))
                        if new_state in seen_states:
                            continue
                        seen_states.add(new_state)

                        z_drop, ratio, progress = _eval_drop(new_state)
                        key = _state_key(new_state, z_drop, ratio, progress)
                        expanded.append((new_state, float(z_drop), float(ratio), float(progress), key))

                if not expanded:
                    break

                expanded.sort(key=lambda x: x[4], reverse=True)
                top = expanded[0]
                if top[4] > best_key:
                    best = (top[0], top[1], top[2], top[3])
                    best_key = top[4]

                if bool(self.cfg.stop_at_first_flip):
                    flips = [x for x in expanded if x[4][0] > 0.5]
                    if flips:
                        best_flip = flips[0]
                        if best_flip[4] > best_key:
                            best = (best_flip[0], best_flip[1], best_flip[2], best_flip[3])
                            best_key = best_flip[4]
                        break

                beam = [x[0] for x in expanded[: max(1, int(self.cfg.beam_width))]]

            selected_best = list(best[0])
            z_cf_best = float(best[1])
            ratio_best = float(best[2])
            progress_best = float(best[3])

            # CoDy floor option: keep CoDy selection if it has better objective for this event.
            if bool(self.cfg.use_cody_floor) and isinstance(cody_row, dict):
                selected_cody = [int(e) for e in list(cody_row.get("selected") or []) if int(e) in set(candidate)]
                z_cf_cody = cody_row.get("counterfactual_prediction")
                if z_cf_cody is None or not np.isfinite(float(z_cf_cody)):
                    z_cf_cody = self._predict_drop(context, candidate, set(selected_cody))
                z_cf_cody = float(z_cf_cody)
                ratio_cody = _cody_delta_ratio(z_orig, z_cf_cody)
                progress_cody = _progress_to_boundary(z_orig, z_cf_cody)
                cody_key = _state_key(tuple(sorted(selected_cody)), z_cf_cody, ratio_cody, progress_cody)
                if cody_key >= best_key:
                    selected_best = list(selected_cody)
                    z_cf_best = float(z_cf_cody)
                    ratio_best = float(ratio_cody)
                    progress_best = float(progress_cody)
                    best_key = cody_key
                    choice = "cody_floor"

        selected_set = {int(e) for e in selected_best}
        z_keep_only = self._predict_keep_only(context, candidate, selected_set)

        # Importance format: selected edges get positive scores, others zero.
        rank_map = {int(e): float(len(selected_best) - i) for i, e in enumerate(selected_best)}
        importance_edges = [float(rank_map.get(int(eid), 0.0)) for eid in candidate]

        elapsed = float(time.perf_counter() - t0)
        extras = {
            "event_idx": int(event_idx) if event_idx is not None else None,
            "candidate_eidx": [int(e) for e in candidate],
            "cf_event_ids": [int(e) for e in selected_best],
            "selected_eidx": [int(e) for e in selected_best],
            "original_prediction": float(z_orig),
            "counterfactual_prediction": float(z_cf_best),
            "prediction_explanation_events_only": float(z_keep_only),
            "achieves_counterfactual_explanation": bool((float(z_orig) >= 0.0) != (float(z_cf_best) >= 0.0)),
            "search_choice": str(choice),
            "search_ratio": float(ratio_best),
            "search_progress": float(progress_best),
            "search_sparsity_penalty": float(self.cfg.sparsity_penalty),
            "search_objective": float(best_key[1] if len(best_key) > 1 else 0.0),
        }

        return ExplanationResult(
            run_id=context.run_id,
            explainer=self.alias,
            context_fp=context.fingerprint(),
            importance_edges=importance_edges,
            importance_nodes=None,
            importance_time=None,
            elapsed_sec=elapsed,
            extras=extras,
        )

print("Model and metric-optimized counterfactual explainer class are ready.")



Model and metric-optimized counterfactual explainer class are ready.


In [ ]:
# Run all trial configs and keep the strongest run on combined metrics.
from datetime import datetime
trial_rows: list[dict[str, Any]] = []
report_by_jsonl: dict[str, dict[str, Any]] = {}


def _extract_behavior_metrics(results_jsonl: Path) -> dict[str, float]:
    rows = []
    with results_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            result = rec.get("result") or {}
            extras = result.get("extras") or {}

            try:
                z_orig = float(extras.get("original_prediction"))
                z_cf = float(extras.get("counterfactual_prediction"))
            except Exception:
                continue

            selected = [int(x) for x in list(extras.get("cf_event_ids") or extras.get("selected_eidx") or [])]
            candidate = [int(x) for x in list(extras.get("candidate_eidx") or [])]
            denom = max(1, len(set(candidate)))
            selected_frac = float(len(set(selected))) / float(denom)

            flip = int((z_orig >= 0.0) != (z_cf >= 0.0))

            keep_same = np.nan
            z_keep = extras.get("prediction_explanation_events_only")
            try:
                z_keep = float(z_keep)
                keep_same = float(int((z_orig >= 0.0) == (z_keep >= 0.0)))
            except Exception:
                pass

            rows.append(
                {
                    "flip": float(flip),
                    "selected_frac": float(selected_frac),
                    "keep_same": float(keep_same) if np.isfinite(keep_same) else np.nan,
                }
            )

    if not rows:
        return {
            "flip_success_rate_proxy": np.nan,
            "first_flip_sparsity_proxy": np.nan,
            "keep_same_rate_proxy": np.nan,
        }

    df = pd.DataFrame(rows)
    flip_rate = float(pd.to_numeric(df["flip"], errors="coerce").mean())

    flip_rows = df[pd.to_numeric(df["flip"], errors="coerce") > 0.5]
    if flip_rows.empty:
        first_flip = 1.0
    else:
        first_flip = float(pd.to_numeric(flip_rows["selected_frac"], errors="coerce").mean())

    keep_same_series = pd.to_numeric(df["keep_same"], errors="coerce")
    keep_same_rate = float(keep_same_series.mean()) if keep_same_series.notna().any() else np.nan

    return {
        "flip_success_rate_proxy": float(flip_rate),
        "first_flip_sparsity_proxy": float(first_flip),
        "keep_same_rate_proxy": float(keep_same_rate),
    }


def _tgnn_official_scores(results_jsonl: Path, *, trial_tag: str) -> dict[str, float]:
    levels = [i / 20.0 for i in range(21)]
    ranking_cfg = {
        "prefer_selected": True,
        "tie_break": "edge_id",
        "uninformative_fallback": "none",
        "natural_support": "nonzero",
    }

    tgnn_dir = out_dir / f"{trial_tag}_tgnn_objective"
    tgnn_dir.mkdir(parents=True, exist_ok=True)

    cfg = EvalConfig(
        out_dir=str(tgnn_dir),
        metrics={
            "tgnn_aufsc": {
                "sparsity_levels": levels,
                "result_as_logit": True,
                "ensure_min_one": True,
                "clamp_non_negative": False,
                "include_series": True,
                "order_strategy": "strict",
                "ranking": dict(ranking_cfg),
            }
        },
        seed=SEED,
        resume=False,
        show_progress=False,
    )

    runner = EvaluationRunner(
        model=model,
        dataset={"events": events},
        extractor=extractor_for_eval,
        explainers=[],
        config=cfg,
    )

    out = runner.compute_metrics_from_results(
        results_path=str(results_jsonl),
        out_dir=str(tgnn_dir),
        resume=False,
    )
    df = pd.read_csv(out["csv"])

    raw_cols = sorted(
        [c for c in df.columns if c.startswith("tgnn_aufsc.fid_inv.@s=")],
        key=lambda c: float(c.split("@s=")[-1]),
    )
    best_cols = sorted(
        [c for c in df.columns if c.startswith("tgnn_aufsc.best.@s=")],
        key=lambda c: float(c.split("@s=")[-1]),
    )

    if not raw_cols or not best_cols:
        return {
            "best_fid": np.nan,
            "aufsc": np.nan,
            "best_fid_raw": np.nan,
            "best_fid_sparsity": np.nan,
            "best_fid_raw_sparsity": np.nan,
        }

    x_best = np.asarray([float(c.split("@s=")[-1]) for c in best_cols], dtype=float)
    y_best = np.asarray([float(pd.to_numeric(df[c], errors="coerce").mean()) for c in best_cols], dtype=float)

    x_raw = np.asarray([float(c.split("@s=")[-1]) for c in raw_cols], dtype=float)
    y_raw = np.asarray([float(pd.to_numeric(df[c], errors="coerce").mean()) for c in raw_cols], dtype=float)

    if y_best.size:
        best_idx = int(np.nanargmax(y_best))
        best_fid = float(y_best[best_idx])
        best_fid_sparsity = float(x_best[best_idx])
    else:
        best_fid = np.nan
        best_fid_sparsity = np.nan

    if y_raw.size:
        raw_idx = int(np.nanargmax(y_raw))
        best_fid_raw = float(y_raw[raw_idx])
        best_fid_raw_sparsity = float(x_raw[raw_idx])
    else:
        best_fid_raw = np.nan
        best_fid_raw_sparsity = np.nan

    aufsc = float(np.trapz(y_best, x_best)) if x_best.size > 1 else float(y_best[0]) if y_best.size else np.nan

    # Consistency check: this should match the metric column generated by the implementation.
    if "tgnn_aufsc.best_fid" in df.columns:
        col_best = float(pd.to_numeric(df["tgnn_aufsc.best_fid"], errors="coerce").mean())
        if np.isfinite(col_best):
            best_fid = col_best

    return {
        "best_fid": best_fid,
        "aufsc": aufsc,
        "best_fid_raw": best_fid_raw,
        "best_fid_sparsity": best_fid_sparsity,
        "best_fid_raw_sparsity": best_fid_raw_sparsity,
    }


def _meets_soft_baseline(metric: float, baseline: float, tol_frac: float = 0.03) -> bool:
    if not (np.isfinite(metric) and np.isfinite(baseline)):
        return False
    return bool(float(metric) >= float(baseline) - float(tol_frac) * abs(float(baseline)))


for cfg_dict in TRIAL_CONFIGS:
    cfg = MetricOptimizedCFConfig(
        alias=str(cfg_dict["alias"]),
        mode=str(cfg_dict.get("mode", "beam")),
        beam_width=int(cfg_dict.get("beam_width", 8)),
        max_steps=int(cfg_dict.get("max_steps", 10)),
        pool_size=int(cfg_dict.get("pool_size", 30)),
        use_cody_floor=bool(cfg_dict.get("use_cody_floor", True)),
        stop_at_first_flip=bool(cfg_dict.get("stop_at_first_flip", False)),
        sparsity_penalty=float(cfg_dict.get("sparsity_penalty", 0.05)),
        cody_seed_bonus=float(cfg_dict.get("cody_seed_bonus", 0.30)),
        progress_weight=float(cfg_dict.get("progress_weight", 0.35)),
        cody_map=baseline_map,
    )
    explainer = MetricOptimizedCounterfactualExplainer(cfg)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = f"{DATASET_NAME}_{MODEL_TYPE}_{cfg.alias}_{ts}"

    cfg_explain = EvalConfig(
        out_dir=str(out_dir),
        metrics={},
        compute_metrics=False,
        save_jsonl=True,
        save_csv=False,
        seed=SEED,
        resume=False,
        show_progress=False,
    )

    runner = EvaluationRunner(
        model=model,
        dataset={"events": events, "dataset_name": DATASET_NAME},
        extractor=extractor_for_eval,
        explainers=[explainer],
        config=cfg_explain,
    )

    out = runner.run(
        anchors,
        k_hop=int(getattr(model, "num_layers", 2) or 2),
        num_neighbors=int(getattr(model, "num_neighbors", 20) or 20),
        run_id=run_id,
        show_progress=True,
    )

    run_dir = Path(out["out_dir"]).resolve()
    out_jsonl = Path(out["jsonl"]).resolve()

    report = compute_cody_style_report(
        results_jsonl=out_jsonl,
        model=model,
        dataset={"events": events},
        score_is_logit=True,
        decision_threshold=0.0,
        max_sparsity=1.0,
        n_grid=101,
        allow_model_inference=False,
        fidelity_mode="score",
        charr_mode="score",
    )
    summary = report.get("summary")
    detail = report.get("detail")

    if summary is None or summary.empty:
        plus = np.nan
        charr = np.nan
    else:
        srow = summary.iloc[0].to_dict()
        plus = float(srow.get("cody_AUFSC_plus", np.nan))
        charr = float(srow.get("cody_CHARR", np.nan))

    if np.isfinite(plus):
        plus_rel = float(plus) / float(max(abs(float(baseline_plus)), 1e-12))
    else:
        plus_rel = np.nan
    if np.isfinite(charr):
        charr_rel = float(charr) / float(max(abs(float(baseline_charr)), 1e-12))
    else:
        charr_rel = np.nan

    behavior = _extract_behavior_metrics(out_jsonl)
    flip_rate = float(behavior["flip_success_rate_proxy"])
    first_flip_proxy = float(behavior["first_flip_sparsity_proxy"])
    keep_same_rate = float(behavior["keep_same_rate_proxy"])

    tgnn_scores = _tgnn_official_scores(out_jsonl, trial_tag=run_id)
    best_fid = float(tgnn_scores.get("best_fid", np.nan))
    aufsc = float(tgnn_scores.get("aufsc", np.nan))
    best_fid_raw = float(tgnn_scores.get("best_fid_raw", np.nan))
    best_fid_sparsity = float(tgnn_scores.get("best_fid_sparsity", np.nan))
    best_fid_raw_sparsity = float(tgnn_scores.get("best_fid_raw_sparsity", np.nan))

    best_fid_rel = (
        float(best_fid) / float(max(abs(float(baseline_best_fid)), 1e-12))
        if np.isfinite(best_fid) and np.isfinite(baseline_best_fid)
        else np.nan
    )
    # AUFSC is signed in this setup; compare as improvement over baseline.
    aufsc_improve = (
        (float(aufsc) - float(baseline_aufsc)) / float(max(abs(float(baseline_aufsc)), 1e-12))
        if np.isfinite(aufsc) and np.isfinite(baseline_aufsc)
        else np.nan
    )

    # Baseline best_fid_raw can be exactly zero; use absolute value fallback then.
    if np.isfinite(best_fid_raw):
        if np.isfinite(baseline_best_fid_raw) and abs(float(baseline_best_fid_raw)) > 1e-9:
            best_fid_raw_rel = float(best_fid_raw) / float(abs(float(baseline_best_fid_raw)))
        else:
            best_fid_raw_rel = float(best_fid_raw)
    else:
        best_fid_raw_rel = np.nan

    # Prefer larger best_fid_sparsity (peak appearing later on the curve) to increase top metrics coverage.
    if np.isfinite(best_fid_sparsity):
        if np.isfinite(baseline_best_fid_sparsity) and abs(float(baseline_best_fid_sparsity)) > 1e-9:
            best_fid_sparsity_rel = float(best_fid_sparsity) / float(abs(float(baseline_best_fid_sparsity)))
        else:
            best_fid_sparsity_rel = float(best_fid_sparsity)
    else:
        best_fid_sparsity_rel = np.nan

    # Upper-metric objective: prioritize TGNN best_fid/aufsc/raw/sparsity, keep CoDy and behavior as secondary terms.
    trial_score = (
        2.40 * float(np.nan_to_num(best_fid_rel, nan=0.0))
        + 2.00 * float(np.nan_to_num(aufsc_improve, nan=0.0))
        + 1.85 * float(np.nan_to_num(best_fid_raw_rel, nan=0.0))
        + 1.55 * float(np.nan_to_num(best_fid_sparsity_rel, nan=0.0))
        + 0.40 * float(np.nan_to_num(plus_rel, nan=0.0))
        + 0.40 * float(np.nan_to_num(charr_rel, nan=0.0))
        + 0.35 * float(np.nan_to_num(flip_rate, nan=0.0))
        + 0.20 * float(np.nan_to_num(1.0 - first_flip_proxy, nan=0.0))
        + 0.20 * float(np.nan_to_num(keep_same_rate, nan=0.0))
    )

    pass_drop = bool(np.isfinite(plus) and np.isfinite(charr) and plus >= 0.94 * baseline_plus and charr >= 0.94 * baseline_charr)
    pass_behavior = bool(np.isfinite(flip_rate) and np.isfinite(first_flip_proxy) and flip_rate >= 0.30 and first_flip_proxy <= 0.55)
    pass_tgnn = _meets_soft_baseline(best_fid, baseline_best_fid, tol_frac=0.02) and _meets_soft_baseline(aufsc, baseline_aufsc, tol_frac=0.02)
    pass_tgnn_upper = pass_tgnn and (
        (np.isfinite(best_fid_sparsity) and np.isfinite(baseline_best_fid_sparsity) and best_fid_sparsity >= 0.90 * baseline_best_fid_sparsity)
        if np.isfinite(baseline_best_fid_sparsity)
        else np.isfinite(best_fid_sparsity)
    )
    pass_trial = bool(pass_tgnn_upper and (pass_drop or pass_behavior))

    trial_row = {
        "trial_name": cfg_dict["name"],
        "alias": cfg.alias,
        "mode": cfg.mode,
        "beam_width": cfg.beam_width,
        "max_steps": cfg.max_steps,
        "pool_size": cfg.pool_size,
        "use_cody_floor": cfg.use_cody_floor,
        "stop_at_first_flip": cfg.stop_at_first_flip,
        "sparsity_penalty": cfg.sparsity_penalty,
        "cody_seed_bonus": cfg.cody_seed_bonus,
        "progress_weight": cfg.progress_weight,
        "run_dir": str(run_dir),
        "out_jsonl": str(out_jsonl),
        "cody_AUFSC_plus": plus,
        "cody_CHARR": charr,
        "plus_rel": plus_rel,
        "charr_rel": charr_rel,
        "flip_success_rate_proxy": flip_rate,
        "first_flip_sparsity_proxy": first_flip_proxy,
        "keep_same_rate_proxy": keep_same_rate,
        "best_fid_official_proxy": best_fid,
        "aufsc_official_proxy": aufsc,
        "best_fid_raw_official_proxy": best_fid_raw,
        "best_fid_sparsity_official_proxy": best_fid_sparsity,
        "best_fid_raw_sparsity_official_proxy": best_fid_raw_sparsity,
        "best_fid_rel": best_fid_rel,
        "aufsc_improve": aufsc_improve,
        "best_fid_raw_rel": best_fid_raw_rel,
        "best_fid_sparsity_rel": best_fid_sparsity_rel,
        "trial_score": trial_score,
        "pass_drop": pass_drop,
        "pass_behavior": pass_behavior,
        "pass_tgnn": pass_tgnn,
        "pass_tgnn_upper": pass_tgnn_upper,
        "pass_trial": pass_trial,
    }
    trial_rows.append(trial_row)
    report_by_jsonl[str(out_jsonl)] = {
        "trial": trial_row,
        "report": report,
        "summary": summary,
        "detail": detail,
    }

    print(
        f"Trial {cfg_dict['name']} -> "
        f"AUFSC+={plus:.6f}, CHARR={charr:.6f}, "
        f"best_fid={best_fid:.6f}, aufsc={aufsc:.6f}, "
        f"best_fid_raw={best_fid_raw:.6f}, best_fid_s={best_fid_sparsity:.3f}, "
        f"flip={flip_rate:.3f}, first_flip={first_flip_proxy:.3f}, keep_same={keep_same_rate:.3f}, "
        f"score={trial_score:.6f}, pass={pass_trial}"
    )

trial_df = pd.DataFrame(trial_rows)
if trial_df.empty:
    raise RuntimeError("No trial runs were produced.")

trial_df["_sort_pass"] = trial_df["pass_trial"].astype(int)
trial_df["_sort_score"] = pd.to_numeric(trial_df["trial_score"], errors="coerce").fillna(-np.inf)
trial_df["_sort_best_fid"] = pd.to_numeric(trial_df["best_fid_official_proxy"], errors="coerce").fillna(-np.inf)
trial_df["_sort_aufsc"] = pd.to_numeric(trial_df["aufsc_official_proxy"], errors="coerce").fillna(-np.inf)
trial_df["_sort_best_fid_raw"] = pd.to_numeric(trial_df["best_fid_raw_official_proxy"], errors="coerce").fillna(-np.inf)
trial_df = trial_df.sort_values(["_sort_pass", "_sort_score", "_sort_best_fid", "_sort_aufsc", "_sort_best_fid_raw"], ascending=False).reset_index(drop=True)

best_trial = trial_df.iloc[0].to_dict()
trial_df_out = trial_df.drop(columns=["_sort_pass", "_sort_score", "_sort_best_fid", "_sort_aufsc", "_sort_best_fid_raw"], errors="ignore")
display(trial_df_out)

best_jsonl = str(best_trial["out_jsonl"])
winning = report_by_jsonl.get(best_jsonl)
if winning is None:
    out_jsonl = Path(best_jsonl).resolve()
    run_dir = Path(best_trial["run_dir"]).resolve()
    report = compute_cody_style_report(
        results_jsonl=out_jsonl,
        model=model,
        dataset={"events": events},
        score_is_logit=True,
        decision_threshold=0.0,
        max_sparsity=1.0,
        n_grid=101,
        allow_model_inference=False,
        fidelity_mode="score",
        charr_mode="score",
    )
    winning = {
        "trial": best_trial,
        "report": report,
        "summary": report.get("summary"),
        "detail": report.get("detail"),
    }

if not bool(best_trial.get("pass_trial", False)):
    print("Warning: no trial satisfied upper-metric criteria; using best available trial.")

run_dir = Path(winning["trial"]["run_dir"]).resolve()
out_jsonl = Path(winning["trial"]["out_jsonl"]).resolve()
base_name = run_dir.name
winning_alias = str(winning["trial"].get("alias", "cf_metric_opt_upper"))

cody_summary = winning["summary"]
cody_detail = winning["detail"]

summary_csv_cody = out_dir / f"{base_name}_cody_metrics_summary.csv"
detail_csv_cody = out_dir / f"{base_name}_cody_metrics_detail.csv"
trials_csv = out_dir / f"{base_name}_trial_search.csv"

if cody_summary is not None and not cody_summary.empty:
    cody_summary.to_csv(summary_csv_cody, index=False)
if cody_detail is not None and not cody_detail.empty:
    cody_detail.to_csv(detail_csv_cody, index=False)
trial_df_out.to_csv(trials_csv, index=False)

print("Winning run_dir:", run_dir)
print("Winning out_jsonl:", out_jsonl)
print("Winning alias:", winning_alias)
print("Saved CoDy summary:", summary_csv_cody)
print("Saved CoDy detail :", detail_csv_cody)
print("Saved trial table :", trials_csv)
print("Baseline targets  :", {
    "cody_AUFSC_plus": baseline_plus,
    "cody_CHARR": baseline_charr,
    "best_fid": baseline_best_fid,
    "aufsc": baseline_aufsc,
    "best_fid_raw": baseline_best_fid_raw,
    "best_fid_sparsity": baseline_best_fid_sparsity,
})

if cody_summary is not None:
    print("Winning CoDy metrics:")
    display(cody_summary)


Evaluation:   0%|          | 0/30 [00:00<?, ?expl/s]

Trial upper_balanced -> AUFSC+=1.776910, CHARR=1.792551, best_fid=-0.887561, aufsc=-1.689001, best_fid_raw=-1.001070, best_fid_s=1.000, flip=0.708, first_flip=0.021, keep_same=0.458, score=1.668982, pass=False


Evaluation:   0%|          | 0/30 [00:00<?, ?expl/s]

Trial upper_bestfid_heavy -> AUFSC+=1.776910, CHARR=1.792551, best_fid=-0.887561, aufsc=-1.689001, best_fid_raw=-1.001070, best_fid_s=1.000, flip=0.708, first_flip=0.021, keep_same=0.458, score=1.668982, pass=False


Evaluation:   0%|          | 0/30 [00:00<?, ?expl/s]

In [ ]:
# Save a compact explanations table for the winning run.
with out_jsonl.open("r", encoding="utf-8") as f:
    _records = [json.loads(line) for line in f if line.strip()]

rows = []
for rec in _records:
    context = rec.get("context") or {}
    target = context.get("target") or {}
    result = rec.get("result") or {}
    extras = result.get("extras") or {}

    candidate = list(extras.get("candidate_eidx") or [])
    selected = list(extras.get("cf_event_ids") or extras.get("selected_eidx") or [])

    rows.append(
        {
            "event_idx": int(target.get("event_idx")) if target.get("event_idx") is not None else pd.NA,
            "explainer": str(result.get("explainer", "unknown")),
            "candidate_size": int(len(candidate)),
            "selected_size": int(len(selected)),
            "achieves_counterfactual_explanation": bool(extras.get("achieves_counterfactual_explanation", False)),
            "search_choice": extras.get("search_choice", pd.NA),
            "original_prediction": pd.to_numeric(extras.get("original_prediction"), errors="coerce"),
            "counterfactual_prediction": pd.to_numeric(extras.get("counterfactual_prediction"), errors="coerce"),
        }
    )

results_df = pd.DataFrame(rows)
out_csv = out_dir / f"{base_name}.csv"
results_df.to_csv(out_csv, index=False)

print("Saved explanations:", out_csv)
display(results_df.head())


In [ ]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


In [ ]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)
